# Machine Learning Assignment 2 — Classification Model Comparison

**Dataset:** UCI Dry Bean  
**Purpose:** Train and compare five classifiers, calculate the six required metrics, save trained pipelines and test data, and generate files for a Streamlit application.

> **Important:** Run every cell from top to bottom in your BITS Virtual Lab. Metrics in the notebook and README must come from your own execution. Do not submit unverified or illustrative results.

## 1. Assignment checklist

This notebook addresses the technical requirements:
- One public classification dataset with at least 12 features and 500 instances.
- Five classifiers: Logistic Regression, Decision Tree, KNN, Naive Bayes, and SVM.
- Six test-set metrics: Accuracy, AUC, Precision, Recall, F1, and MCC.
- A consistent stratified train/test split.
- Saved models, `test_data.csv`, `app.py`, `requirements.txt`, and a README draft.
- A comparison table and confusion matrices/classification reports.

You must still create your own GitHub repository, deploy the app, and capture the required screenshot while working in BITS Virtual Lab.

In [ ]:
# Install dependencies if they are not already installed in your notebook environment.
%pip install -q pandas numpy scikit-learn matplotlib seaborn joblib requests streamlit tabulate

In [ ]:
# Import the libraries needed for data handling, training, evaluation, visualization, and saving files.
from pathlib import Path
import io
import json
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import requests

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
PROJECT_DIR = Path("ml_assignment_project")
MODEL_DIR = PROJECT_DIR / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load the UCI Dry Bean dataset

The Dry Bean dataset is a public multiclass classification dataset from the UCI Machine Learning Repository. It contains numeric measurements of bean images and a target class identifying the bean variety. This dataset is a suitable starting choice because it exceeds the assignment's minimum size and feature requirements.

The following cell downloads the official Excel file. If your BITS Virtual Lab blocks the download, download the dataset manually from the UCI Dry Bean page and update `DATA_PATH` below.

In [ ]:
# Download the public dataset from the UCI repository.
DATA_URL = "https://archive.ics.uci.edu/static/public/602/dry+bean+dataset.zip"
DATA_PATH = Path("Dry_Bean_Dataset.xlsx")

if not DATA_PATH.exists():
    response = requests.get(DATA_URL, timeout=60)
    response.raise_for_status()
    zip_path = Path("dry_bean_dataset.zip")
    zip_path.write_bytes(response.content)
    with zipfile.ZipFile(zip_path, "r") as zf:
        excel_files = [name for name in zf.namelist() if name.lower().endswith(".xlsx")]
        if not excel_files:
            raise FileNotFoundError("The downloaded UCI archive did not contain an Excel file.")
        extracted = zf.extract(excel_files[0], ".")
        DATA_PATH = Path(extracted)

# Read the dataset. The final column contains the bean class label.
df = pd.read_excel(DATA_PATH)
df.head()

In [ ]:
# Inspect dataset dimensions, data types, missing values, and target-class distribution.
print(f"Number of rows (instances): {df.shape[0]}")
print(f"Number of columns including target: {df.shape[1]}")
print(f"Number of input features: {df.shape[1] - 1}")
print("\nColumn data types:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing values per column:")
display(df.isna().sum().to_frame("missing_values"))
target_column = df.columns[-1]
print(f"\nTarget column: {target_column}")
display(df[target_column].value_counts().to_frame("class_count"))

In [ ]:
# Validate the assignment's minimum dataset-size and feature-count requirements.
assert df.shape[0] >= 500, "Dataset must contain at least 500 instances."
assert df.shape[1] - 1 >= 12, "Dataset must contain at least 12 input features."
assert df[target_column].nunique() >= 2, "The target must contain at least two classes."

# Separate features from the target, and encode class labels as integers for consistent metric handling.
X = df.drop(columns=[target_column]).copy()
y_encoder = LabelEncoder()
y = y_encoder.fit_transform(df[target_column].astype(str))
class_names = y_encoder.classes_.tolist()

print("Dataset requirements passed.")
print("Input feature count:", X.shape[1])
print("Class names:", class_names)

## 3. Create a consistent train/test split

All five models use the same stratified split. Stratification helps preserve the class proportions in both training and test data. Scaling is included inside pipelines for models that are sensitive to feature scale.

In [ ]:
# Split once so each classifier is evaluated against exactly the same held-out examples.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Feature count:", X_train.shape[1])

## 4. Define the five required classifiers

- **Logistic Regression:** scaled numeric features and a higher iteration limit for convergence.
- **Decision Tree:** interpretable rule-based classifier.
- **KNN:** scaled features because distance calculations are scale-sensitive.
- **Gaussian Naive Bayes:** suitable for continuous numeric features.
- **SVM:** scaled features and probability estimates for AUC calculation.

Imputation is included in each pipeline as a defensive preprocessing step. Models are trained only in this notebook, not in the Streamlit app.

In [ ]:
# Build pipelines so preprocessing is saved together with each fitted classifier.
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "KNN": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Naive Bayes": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", GaussianNB())
    ]),
    "SVM": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))
    ])
}
print("Models prepared:", list(models.keys()))

## 5. Train and evaluate all models

For multiclass metrics, Precision, Recall, and F1 use **weighted averaging** so each class contributes according to its number of true examples. AUC is calculated using one-vs-rest probabilities and macro averaging. The exact averaging method is stated here so the comparison is transparent and consistent.

In [ ]:
# Calculate multiclass AUC from predicted probabilities, with a safe fallback if a model cannot provide them.
def calculate_auc(fitted_model, X_eval, y_true, number_of_classes):
    if hasattr(fitted_model, "predict_proba"):
        scores = fitted_model.predict_proba(X_eval)
    elif hasattr(fitted_model, "decision_function"):
        scores = fitted_model.decision_function(X_eval)
    else:
        return np.nan

    try:
        if number_of_classes == 2:
            if np.ndim(scores) == 2:
                return roc_auc_score(y_true, scores[:, 1])
            return roc_auc_score(y_true, scores)
        return roc_auc_score(
            y_true, scores, multi_class="ovr", average="macro", labels=np.arange(number_of_classes)
        )
    except (ValueError, TypeError):
        return np.nan

# Fit each classifier, predict the held-out test set, and record all six required metrics.
results = []
fitted_models = {}
predictions = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    fitted_models[model_name] = model
    predictions[model_name] = y_pred

    metric_row = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": calculate_auc(model, X_test, y_test, len(class_names)),
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }
    results.append(metric_row)

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.style.format("{:.4f}"))

In [ ]:
# Visualize the metric comparison. These values are generated from the current execution.
ax = results_df.plot(kind="bar", figsize=(13, 6), ylim=(0, 1.05))
ax.set_title("Comparison of Five Classifiers on the Held-Out Test Set")
ax.set_ylabel("Metric score")
ax.set_xlabel("Classifier")
plt.xticks(rotation=20, ha="right")
plt.legend(loc="lower right", ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
# Display a confusion matrix and classification report for each classifier.
for model_name, y_pred in predictions.items():
    print(f"\n{'=' * 70}\n{model_name}\n{'=' * 70}")
    print(classification_report(
        y_test, y_pred, labels=np.arange(len(class_names)),
        target_names=class_names, zero_division=0
    ))
    cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(class_names)))
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix — {model_name}")
    plt.xlabel("Predicted class")
    plt.ylabel("Actual class")
    plt.tight_layout()
    plt.show()

## 6. Write evidence-based model observations

The generated text below gives a starting point based on the measured scores. Review the confusion matrices and the relative scores before using any observations in the final README. Add dataset-specific reasoning in your own words; do not claim that a model overfits unless you have evidence for it.

In [ ]:
# Rank models by weighted F1 as a primary summary, while retaining all six metrics for the final decision.
ranked = results_df.sort_values(["F1 Score", "MCC", "AUC"], ascending=False)
winner = ranked.index[0]
print("Provisional overall winner based on weighted F1, with MCC and AUC as tie-breakers:", winner)

for model_name, row in results_df.iterrows():
    print(
        f"{model_name}: Accuracy={row['Accuracy']:.4f}, AUC={row['AUC']:.4f}, "
        f"Precision={row['Precision']:.4f}, Recall={row['Recall']:.4f}, "
        f"F1={row['F1 Score']:.4f}, MCC={row['MCC']:.4f}."
    )

## 7. Save trained models, test data, and metric results

The saved model files are complete pipelines, including preprocessing where applicable. The test CSV contains the held-out examples with the target column so the app can calculate metrics. Keep this labelled test data separate from training data.

In [ ]:
# Save each fitted pipeline under a stable filename for use by the Streamlit app.
filename_map = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "SVM": "svm.joblib"
}
for model_name, fitted_model in fitted_models.items():
    joblib.dump(fitted_model, MODEL_DIR / filename_map[model_name])

# Save the held-out test examples, including their original readable target labels.
test_output = X_test.copy()
test_output[target_column] = y_encoder.inverse_transform(y_test)
test_output.to_csv(PROJECT_DIR / "test_data.csv", index=False)

# Save the measured comparison table and class-label metadata.
results_df.to_csv(PROJECT_DIR / "model_comparison.csv")
with open(PROJECT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump({
        "target_column": target_column,
        "feature_columns": X.columns.tolist(),
        "class_names": class_names,
        "model_filenames": filename_map,
        "metric_averaging": "weighted precision/recall/F1; macro one-vs-rest AUC"
    }, f, indent=2)

print("Saved models, test data, metrics, and metadata under:", PROJECT_DIR.resolve())

## 8. Generate the Streamlit application

The application loads the saved pipelines. It does **not** train models. Upload a labelled CSV with the same feature columns and target column as the dataset; the app validates the input before evaluating it.

In [ ]:
# Write the Streamlit application file into the project directory.
app_source = r"""
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import streamlit as st
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, classification_report
)

st.set_page_config(page_title="Classifier Comparison", page_icon="📊", layout="wide")
st.title("Machine Learning Classifier Comparison")
st.write("Upload labelled test data to evaluate the saved classifiers. Models are loaded from disk; no retraining occurs.")

ROOT = Path(__file__).parent
MODEL_DIR = ROOT / "model"
with open(ROOT / "metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

MODEL_FILES = metadata["model_filenames"]
FEATURES = metadata["feature_columns"]
TARGET = metadata["target_column"]
CLASS_NAMES = metadata["class_names"]
LABEL_TO_INT = {label: i for i, label in enumerate(CLASS_NAMES)}

@st.cache_resource
def load_models():
    return {name: joblib.load(MODEL_DIR / filename) for name, filename in MODEL_FILES.items()}

models = load_models()
uploaded = st.file_uploader("Upload a labelled test CSV", type=["csv"])
if uploaded is not None:
    data = pd.read_csv(uploaded)
    missing = [c for c in FEATURES + [TARGET] if c not in data.columns]
    if missing:
        st.error(f"Missing required columns: {missing}")
        st.stop()
    if data.empty:
        st.error("The uploaded CSV contains no rows.")
        st.stop()

    unknown_labels = sorted(set(data[TARGET].astype(str)) - set(CLASS_NAMES))
    if unknown_labels:
        st.error(f"Target column contains unknown class labels: {unknown_labels}")
        st.stop()

    X_eval = data[FEATURES].copy()
    y_eval = data[TARGET].astype(str).map(LABEL_TO_INT).to_numpy()
    selected = st.selectbox("Select a classifier", ["Compare all models"] + list(models.keys()))

    def auc_score(model, X, y):
        try:
            scores = model.predict_proba(X)
            if len(CLASS_NAMES) == 2:
                return roc_auc_score(y, scores[:, 1])
            return roc_auc_score(y, scores, multi_class="ovr", average="macro",
                                 labels=np.arange(len(CLASS_NAMES)))
        except (ValueError, TypeError, AttributeError):
            return float("nan")

    def evaluate(name, model):
        pred = model.predict(X_eval)
        return {
            "Model": name,
            "Accuracy": accuracy_score(y_eval, pred),
            "AUC": auc_score(model, X_eval, y_eval),
            "Precision": precision_score(y_eval, pred, average="weighted", zero_division=0),
            "Recall": recall_score(y_eval, pred, average="weighted", zero_division=0),
            "F1 Score": f1_score(y_eval, pred, average="weighted", zero_division=0),
            "MCC": matthews_corrcoef(y_eval, pred)
        }, pred

    if selected == "Compare all models":
        rows = []
        prediction_map = {}
        for name, model in models.items():
            row, pred = evaluate(name, model)
            rows.append(row)
            prediction_map[name] = pred
        table = pd.DataFrame(rows).set_index("Model")
        st.subheader("All-model comparison")
        st.dataframe(table.style.format("{:.4f}"), use_container_width=True)
        chosen_for_detail = st.selectbox("Show detailed report for", list(models.keys()))
        pred = prediction_map[chosen_for_detail]
        current_model = models[chosen_for_detail]
    else:
        row, pred = evaluate(selected, models[selected])
        table = pd.DataFrame([row]).set_index("Model")
        st.subheader(f"Metrics — {selected}")
        st.dataframe(table.style.format("{:.4f}"), use_container_width=True)
        chosen_for_detail = selected
        current_model = models[selected]

    st.subheader(f"Confusion matrix — {chosen_for_detail}")
    labels = np.arange(len(CLASS_NAMES))
    cm = confusion_matrix(y_eval, pred, labels=labels)
    cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
    st.dataframe(cm_df, use_container_width=True)

    st.subheader(f"Classification report — {chosen_for_detail}")
    report = classification_report(y_eval, pred, labels=labels, target_names=CLASS_NAMES,
                                  zero_division=0, output_dict=True)
    st.dataframe(pd.DataFrame(report).transpose(), use_container_width=True)
else:
    st.info("Upload a labelled CSV containing the feature columns and target column to begin.")
    st.caption("Use the project's test_data.csv as a sample input.")
"""
(PROJECT_DIR / "app.py").write_text(app_source, encoding="utf-8")
print("Created:", PROJECT_DIR / "app.py")

In [ ]:
# Create a requirements file for local execution and Streamlit Community Cloud deployment.
requirements = '''streamlit
scikit-learn
numpy
pandas
matplotlib
seaborn
joblib
requests
openpyxl
tabulate
'''
(PROJECT_DIR / "requirements.txt").write_text(requirements, encoding="utf-8")
print((PROJECT_DIR / "requirements.txt").read_text())

## 9. Generate a README draft

This file is populated with the actual comparison results after the model-training cells have run. Review and customize the observations before submitting. The GitHub and live-app URLs are placeholders until you create and deploy your own project.

In [ ]:
# Build the README with measured results; replace links and refine observations after deployment.
repo_url = "PASTE_YOUR_GITHUB_REPOSITORY_URL_HERE"
app_url = "PASTE_YOUR_LIVE_STREAMLIT_APP_URL_HERE"

table_md = results_df.to_markdown(floatfmt=".4f")
observation_lines = []
for model_name, row in results_df.iterrows():
    observation_lines.append(
        f"| {model_name} | Accuracy {row['Accuracy']:.4f}; AUC {row['AUC']:.4f}; "
        f"Precision {row['Precision']:.4f}; Recall {row['Recall']:.4f}; "
        f"F1 {row['F1 Score']:.4f}; MCC {row['MCC']:.4f} |"
    )

readme = f'''# Machine Learning Assignment 2 — Classifier Comparison

## Problem statement
This project compares five supervised classification algorithms on the public UCI Dry Bean dataset. The goal is to predict the bean variety from numeric measurements and evaluate all classifiers on the same held-out test set.

## Dataset description
- **Source:** UCI Machine Learning Repository — Dry Bean Dataset
- **Instances used:** {len(df)}
- **Input features:** {X.shape[1]}
- **Target column:** `{target_column}`
- **Classes:** {", ".join(class_names)}
- **Task:** Multiclass classification
- **Split:** 80% training / 20% test, stratified, random state {RANDOM_STATE}

The input features are numeric measurements extracted from bean images. The target label represents the bean variety. Check the official UCI dataset page for the detailed description of individual features.

## Repository and live application
- **GitHub repository:** {repo_url}
- **Live Streamlit app:** {app_url}

## Models and evaluation metrics
Precision, Recall, and F1 use weighted averaging. AUC uses macro one-vs-rest multiclass AUC. The values below were calculated from the held-out test set by this notebook.

{table_md}

## Model performance observations
| Model | Observation based on measured test-set results |
|---|---|
{chr(10).join(observation_lines)}

Review the table and confusion matrices before final submission. Explain the meaningful differences between the models in your own words, including any class-specific errors. Do not infer overfitting from test metrics alone; compare training and test performance if you want to make an overfitting claim.

## Overall winner
**Provisional winner by weighted F1, using MCC and AUC as tie-breakers:** {winner}

This is a rule for selecting a provisional winner, not a universal claim that the same model is best for every objective. Review all six metrics and the confusion matrices before finalizing your conclusion.

## How to run
1. Install packages from `requirements.txt`.
2. Run the notebook from top to bottom to train models and create the project artifacts.
3. From this folder, run `streamlit run app.py`.
4. Upload a labelled CSV with the same feature columns and target column.

## Repository contents
- `app.py` — Streamlit evaluation interface
- `requirements.txt` — dependencies
- `README.md` — project documentation
- `test_data.csv` — labelled held-out test data
- `model/` — saved trained pipelines
- `metadata.json` — feature, target, class, and model-file metadata
- `model_comparison.csv` — measured comparison results

## Important limitations
The app evaluates labelled test data. It does not train or update models on uploaded data. Metric values can change if you change the dataset version, split, or model settings.
'''
(PROJECT_DIR / "README.md").write_text(readme, encoding="utf-8")
print("README draft created. GitHub and app links are placeholders to replace after deployment.")

In [ ]:
# Verify the expected project files and model artifacts were created.
expected_files = [
    PROJECT_DIR / "app.py",
    PROJECT_DIR / "requirements.txt",
    PROJECT_DIR / "README.md",
    PROJECT_DIR / "test_data.csv",
    PROJECT_DIR / "metadata.json",
    PROJECT_DIR / "model_comparison.csv",
]
for path in expected_files:
    print(("OK      " if path.exists() else "MISSING "), path)

print("\nSaved model files:")
for path in sorted(MODEL_DIR.glob("*.joblib")):
    print("OK      ", path)

## 10. Local app test and final submission checklist

Run the following command from the project folder in a terminal:

```bash
streamlit run app.py
```

Then upload `test_data.csv` and confirm that the app displays the six metrics, the all-model comparison, confusion matrix, and classification report.

Before final submission:
1. Customize the README observations using your actual outputs.
2. Create a GitHub repository and push the complete project folder.
3. Deploy `app.py` on Streamlit Community Cloud and test the live link.
4. Capture a genuine screenshot showing execution in BITS Virtual Lab.
5. Create one PDF containing, in the assignment's required order: GitHub URL, live Streamlit URL, BITS Virtual Lab screenshot, and the complete README.
6. Verify every link and ensure the final PDF is submitted before the deadline.

**Do not fabricate the screenshot or links.** The GitHub repository, live deployment, and BITS Virtual Lab evidence must be completed by you.